In [0]:
pip install torch

In [0]:
pip install tqdm

In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq
from pyspark.storagelevel import *
from math import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.pipeline import PipelineModel

### LSTM Experiment

In [0]:
lstm = spark.read.table("hive_metastore.default.lreg_df")

In [0]:
lstm = lstm.drop("AO","NOME","TIPOINST","TAGCOM","REDE","ID_prefix","ID_OBJECTO")

In [0]:
lstm = lstm.withColumn("AMPM_flag", when(col("AM_PM")=="PM", 1.0).otherwise(0.0))

In [0]:
NUMERIC_COLS = [
     "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
     "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
     "EVENT_COUNT_I","EVENT_COUNT_T",
     "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
     "DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY", "DAY_OF_WEEK",
     "AMPM_flag"
                ]

CAT_COLS = ["ID", "CONCELHO"]

CUT_OFF    = "2023-11-30"   
SAVE_PATH  = "dbfs:/models/lstm_indexers_v1"      # where to persist indexers + metadata
NSTEPS     = 96


In [0]:
ID_COL = "ID"
LABEL_COL = "has_falha"   # we'll rename to "label"

SELECT_COLS = (
    [ID_COL, "DATE", LABEL_COL] +
    [c for c in CAT_COLS if c != ID_COL] +
    NUMERIC_COLS
)

base0 = (
    lstm
    .select(*[F.col(c) for c in SELECT_COLS])
    .withColumn("DATE", F.col("DATE").cast("timestamp"))
    .withColumnRenamed(LABEL_COL, "label")
)
display(base0.limit(10))


# Cyclical encodings
pi = 3.141592653589793
df_time = (base0
    .withColumn("DOW_SIN", F.sin(2*pi * (F.col("DAY_OF_WEEK")/7.0)))
    .withColumn("DOW_COS", F.cos(2*pi * (F.col("DAY_OF_WEEK")/7.0)))
    .withColumn("HOUR_SIN", F.sin(2*pi * (F.col("HOUR_OF_DAY")/24.0)))
    .withColumn("HOUR_COS", F.cos(2*pi * (F.col("HOUR_OF_DAY")/24.0)))
    .withColumn("DOM_SIN",  F.sin(2*pi * (F.col("DAY_OF_MONTH")/31.0)))
    .withColumn("DOM_COS",  F.cos(2*pi * (F.col("DAY_OF_MONTH")/31.0)))
    .withColumn("DOY_SIN",  F.sin(2*pi * (F.col("DAY_OF_YEAR")/365.0)))
    .withColumn("DOY_COS",  F.cos(2*pi * (F.col("DAY_OF_YEAR")/365.0)))
    # drop originals that now have cyclical counterparts
    .drop("DAY_OF_WEEK","HOUR_OF_DAY","DAY_OF_MONTH","DAY_OF_YEAR")
)

display(df_time.limit(20))

In [0]:
train_df = df_time.filter(F.col("DATE") < F.lit(CUT_OFF))

def build_indexer_pipeline(cat_cols):
    stages = []
    for c in cat_cols:
        stages.append(
            StringIndexer(
                inputCol=c, outputCol=f"{c}_INDEX",
                handleInvalid="keep",
                stringOrderType="frequencyDesc"
            )
        )
    return Pipeline(stages=stages)

# Load if exists; else fit on TRAIN and save
# try:
#     indexer_model = PipelineModel.load(SAVE_PATH)
#     print(f"Loaded indexer pipeline from {SAVE_PATH}")
# except Exception:
print("No saved indexers found. Fitting on TRAIN split…")
indexer_model = build_indexer_pipeline(CAT_COLS).fit(train_df)
indexer_model.write().overwrite().save(SAVE_PATH)
print(f"Saved indexer pipeline to {SAVE_PATH}")

df_idx = indexer_model.transform(df_time)

# Peek mappings
for st in indexer_model.stages:
    cname = st.getInputCol()
    labs = list(st.labels)[:12]
    print(f"{cname} → {cname}_INDEX | sample:", list(enumerate(labs)))

display(df_idx.select("ID","ID_INDEX","CONCELHO","CONCELHO_INDEX").limit(20))


In [0]:
# Numeric final = base numerics minus H/D/M/Y plus cyclical pairs
NUMERIC_FINAL = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T", "AMPM_flag",
    "TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "HOUR_SIN","HOUR_COS","DOW_SIN", "DOW_COS","DOM_SIN","DOM_COS","DOY_SIN","DOY_COS"
]

CAT_IDX_COLS = [f"{c}_INDEX" for c in CAT_COLS]

FEATURES_FOR_LSTM = NUMERIC_FINAL + CAT_IDX_COLS
N_NUM = len(NUMERIC_FINAL)
N_CAT = len(CAT_IDX_COLS)
NFEAT = len(FEATURES_FOR_LSTM)
print("Numeric dims:", N_NUM, "| Categorical count:", N_CAT, "| Total INPUT_SIZE:", NFEAT)


# === SCALE NUMERIC FEATURES (fit on TRAIN only, apply to ALL) ===
# keep names identical; overwrite columns in-place on df_idx
from pyspark.sql import functions as F

# do NOT scale these (binary/cyclical already bounded)
DO_NOT_SCALE = {"AMPM_flag","HOUR_SIN","HOUR_COS","DOW_SIN","DOW_COS","DOM_SIN","DOM_COS","DOY_SIN","DOY_COS"}

# scale the rest of NUMERIC_FINAL
NUMERIC_TO_SCALE = [c for c in NUMERIC_FINAL if c not in DO_NOT_SCALE]

# (optional) treat heavy-tailed counts via log1p + zscore
LOG1P_THEN_SCALE = [c for c in ["EVENT_COUNT_I","EVENT_COUNT_T"] if c in NUMERIC_TO_SCALE]
NUMERIC_TO_SCALE = [c for c in NUMERIC_TO_SCALE if c not in LOG1P_THEN_SCALE]

# fit stats on TRAIN split only
train_df_for_stats = df_idx.filter(F.col("DATE") < F.lit(CUT_OFF))
agg_exprs = []
# plain zscore stats
agg_exprs += [F.avg(c).alias(f"{c}__mu") for c in NUMERIC_TO_SCALE]
agg_exprs += [F.stddev_pop(c).alias(f"{c}__sd") for c in NUMERIC_TO_SCALE]
# log1p zscore stats (mean & std of transformed values)
agg_exprs += [F.avg(F.log1p(F.col(c))).alias(f"{c}__mu_log1p") for c in LOG1P_THEN_SCALE]
agg_exprs += [F.stddev_pop(F.log1p(F.col(c))).alias(f"{c}__sd_log1p") for c in LOG1P_THEN_SCALE]

stats = train_df_for_stats.select(*agg_exprs).first().asDict()

def _safe(v, default=1.0):
    return float(v) if v not in (None, 0.0) else float(default)

# apply to FULL data (overwrite columns; names unchanged)
for c in NUMERIC_TO_SCALE:
    mu = float(stats.get(f"{c}__mu", 0.0))
    sd = _safe(stats.get(f"{c}__sd"), 1.0)
    df_idx = df_idx.withColumn(c, (F.col(c) - F.lit(mu)) / F.lit(sd))

for c in LOG1P_THEN_SCALE:
    mu = float(stats.get(f"{c}__mu_log1p", 0.0))
    sd = _safe(stats.get(f"{c}__sd_log1p"), 1.0)
    df_idx = df_idx.withColumn(c, (F.log1p(F.col(c)) - F.lit(mu)) / F.lit(sd))
# === END SCALING ===



# Compute cardinalities (max index + 1) on TRAIN to size embeddings safely
maxes = (df_idx.filter(F.col("DATE") < F.lit(CUT_OFF))
              .agg(*[F.max(c).alias(c) for c in CAT_IDX_COLS])
              .collect()[0])

cat_cardinalities = {}
for c in CAT_COLS:
    max_idx = maxes[f"{c}_INDEX"] if maxes[f"{c}_INDEX"] is not None else -1.0
    # +1 to convert max index to size; +1 more for safety if desired (UNK is already handled by StringIndexer.keep)
    cat_cardinalities[c] = int(max_idx) + 1

print("Cat cardinalities:", cat_cardinalities)

# Persist metadata for the PyTorch stage
import json, os

cat_cards_to_save = {f"{k}_INDEX": int(v) for k, v in cat_cardinalities.items()}

META_DIR  = "dbfs:/models/lstm_indexers_v1"
META_DIR_LOCAL = META_DIR.replace("dbfs:/", "/dbfs/")
dbutils.fs.mkdirs(META_DIR)                  # create in DBFS view
os.makedirs(META_DIR_LOCAL, exist_ok=True)   # ensure local mount exists

dbfs_meta = f"{META_DIR_LOCAL}/meta.json"

with open(dbfs_meta, "w") as f:
    json.dump({
        "FEATURES_FOR_LSTM": FEATURES_FOR_LSTM,
        "N_NUM": N_NUM,
        "N_CAT": N_CAT,
        "CAT_COLS": CAT_COLS,
        "CAT_IDX_COLS": CAT_IDX_COLS,
        "cat_cardinalities": cat_cards_to_save   # <-- save with _INDEX keys
    }, f, indent=2)
print("Saved meta to", dbfs_meta)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

NSTEPS = 96

# Assemble per-row feature array (double)
feat_expr = F.array(*[F.col(c).cast("double") for c in FEATURES_FOR_LSTM])

dfF = (df_idx
       .select("ID","DATE","label", feat_expr.alias("f"))
       .withColumn("ts", F.col("DATE").cast("long")))

w = Window.partitionBy("ID").orderBy("ts").rowsBetween(-NSTEPS+1, 0)

df_seq = (dfF.withColumn("features", F.collect_list("f").over(w))
              .where(F.size("features") == NSTEPS)
              .select("ID","DATE","label","features"))

# Split
df_train = df_seq.filter(F.col("DATE") < F.lit(CUT_OFF)).select("features","label")
df_test  = df_seq.filter(F.col("DATE") >= F.lit(CUT_OFF)).select("features","label")

# Sanity checks (Databricks-preferred)
display(df_train.select(F.size("features").alias("T"), F.size(F.col("features")[0]).alias("F")).summary())
display(df_test.select(F.size("features").alias("T"), F.size(F.col("features")[0]).alias("F")).summary())

# Write Parquet
spark.conf.set("spark.sql.files.maxRecordsPerFile", 400000)
TRAIN_PATH = f"dbfs:/lstm_train_96x{NFEAT}"
TEST_PATH  = f"dbfs:/lstm_test_96x{NFEAT}"

(df_train.repartition(200).write.mode("overwrite").option("compression","snappy").parquet(TRAIN_PATH))
(df_test.repartition(200).write.mode("overwrite").option("compression","snappy").parquet(TEST_PATH))
print("Wrote:", TRAIN_PATH, "and", TEST_PATH)


In [0]:
display(df_train)

In [0]:
import os, json, math, numpy as np, sys
import builtins
import torch, torch.nn as nn, torch.optim as optim
import pyarrow.dataset as ds
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score
os.environ["TQDM_NOTEBOOK"] = "0"   # force std text bar
from tqdm import tqdm
import gc, time, math

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
META_PATH = "/dbfs/models_lstm_indexers_v1/meta.json"

def to_local_dbfs(p: str) -> str:
    return p.replace("dbfs:/", "/dbfs/")

TRAIN_PATH_LOCAL = to_local_dbfs(TRAIN_PATH)
TEST_PATH_LOCAL  = to_local_dbfs(TEST_PATH)
print(TRAIN_PATH_LOCAL, TEST_PATH_LOCAL)

# NEW: optionally use a validation set if you wrote one (e.g., dbfs:/lstm_val_96x{NFEAT})
# If it doesn't exist, we'll silently fall back to TEST for per-epoch evaluation.
VAL_PATH = f"dbfs:/lstm_val_96x{NFEAT}"
VAL_PATH_LOCAL = to_local_dbfs(VAL_PATH)
import os
USE_VAL = os.path.exists(VAL_PATH_LOCAL)
if USE_VAL:
    print("Using VAL for per-epoch metrics:", VAL_PATH_LOCAL)
else:
    print("VAL not found; evaluating on TEST each epoch.")

def count_rows_local(parquet_path_local: str) -> int:
    dset = ds.dataset(parquet_path_local, format="parquet")
    return builtins.sum(frag.count_rows() for frag in dset.get_fragments())


# Load meta
with open("/dbfs/models_lstm_indexers_v1/meta.json","r") as f:
    META = json.load(f)

CAT_IDX_COLS = META["CAT_IDX_COLS"]
cat_cards     = META["cat_cardinalities"]

# Normalize keys to match CAT_IDX_COLS (append _INDEX if missing)
cat_cards_idx = { (k if k.endswith("_INDEX") else f"{k}_INDEX"): int(v)
                  for k, v in cat_cards.items() }

def emb_dim(n: int) -> int:
    n = int(n)
    # safe built-ins to avoid Spark shadowing
    val = int(math.sqrt(n)) * 2
    val = builtins.max(2, builtins.min(64, val))
    return val

# Sanity check
missing = [k for k in CAT_IDX_COLS if k not in cat_cards_idx]
if missing:
    raise KeyError(f"Missing cardinalities for: {missing}. Have keys: {list(cat_cards_idx.keys())}")

EMB_SPECS = [(name, cat_cards_idx[name], emb_dim(cat_cards_idx[name])) for name in CAT_IDX_COLS]
print("EMB_SPECS:", EMB_SPECS)

FEATURES_FOR_LSTM = META["FEATURES_FOR_LSTM"]
N_NUM   = int(META["N_NUM"])
CAT_IDX_COLS = META["CAT_IDX_COLS"]          # e.g. ["ID_INDEX","CONCELHO_INDEX"]
cat_cards     = META["cat_cardinalities"]     # dict with sizes
INPUT_SIZE    = len(FEATURES_FOR_LSTM)

HIDDEN, LAYERS, DROPOUT = 128, 1, 0.1
LR, BATCH_SIZE, EPOCHS, PATIENCE = 2e-3, 128, 8, 2
SCAN_BATCH = 2048

def minibatches_from_parquet(folder, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=True):
    dataset = ds.dataset(folder, format="parquet")
    frags = list(dataset.get_fragments())
    if shuffle_files: np.random.shuffle(frags)
    for frag in frags:
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=scan_batch)
        for rec in scanner.to_reader():
            X = np.array(rec["features"].to_pylist(), dtype=np.float32)  # (B,96,INPUT_SIZE)
            y = np.array(rec["label"].to_pylist(),    dtype=np.float32)
            n = X.shape[0]; idx = np.arange(n); np.random.shuffle(idx)
            for i in range(0, n, batch_size):
                j = idx[i:i+batch_size]
                yield X[j], y[j]


class LSTMWithEmb(nn.Module):
    def __init__(self, n_num, emb_specs, hidden=128, layers=1, dropout=0.1, bidirectional=False):
        super().__init__()
        self.emb_names = [n for n,_,_ in emb_specs]
        self.embs = nn.ModuleDict({n: nn.Embedding(card, dim) for n,card,dim in emb_specs})
        # use Python's sum, not Spark's
        emb_total = builtins.sum([dim for _,_,dim in emb_specs])
        in_size = n_num + emb_total
        self.lstm = nn.LSTM(in_size, hidden, num_layers=layers,
                            dropout=(dropout if layers>1 else 0.0),
                            batch_first=True, bidirectional=bidirectional)
        out = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.LayerNorm(out), nn.Linear(out, 1))

    def forward(self, x):
        x_num = x[:, :, :N_NUM]               # numeric block
        x_cat = x[:, :, N_NUM:]               # stacked cat indices as floats
        cat_embs = []
        for i, name in enumerate(self.emb_names):
            idx = x_cat[:, :, i].long().clamp(min=0)
            cat_embs.append(self.embs[name](idx))
        if cat_embs:
            xin = torch.cat([x_num, torch.cat(cat_embs, dim=-1)], dim=-1)
        else:
            xin = x_num
        _, (hn, _) = self.lstm(xin)
        last = hn[-1]
        return self.head(last).squeeze(1)     # logits

model = LSTMWithEmb(N_NUM, EMB_SPECS, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

def count_rows_local(parquet_path_local: str) -> int:
    dset = ds.dataset(parquet_path_local, format="parquet")
    return builtins.sum(frag.count_rows() for frag in dset.get_fragments())

# Optional (nice for % complete)

n_train = None
try:
    n_train = count_rows_local(TRAIN_PATH_LOCAL)
except Exception:
    pass
est_batches = int(math.ceil(n_train / BATCH_SIZE)) if n_train else None

BEST_AUC, BAD = -1.0, 0
for ep in range(1, EPOCHS+1):
    model.train()
    bar = tqdm(total=est_batches or 0, desc=f"Epoch {ep}/{EPOCHS}", leave=False, ncols=0, file=sys.stdout)
    b = 0
    for X, y in minibatches_from_parquet(TRAIN_PATH_LOCAL, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=True):
        Xt = torch.from_numpy(X).to(DEVICE, non_blocking=True)
        yt = torch.from_numpy(y).to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(Xt), yt)
        loss.backward(); opt.step()
        b += 1
        if est_batches: bar.update(1)
        if (b % 50) == 0:
            del Xt, yt
            import gc; gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
    bar.close()

    # evaluation progress bar (by fragments)
    model.eval()
    frag_list = list(ds.dataset(TEST_PATH_LOCAL, format="parquet").get_fragments())
    eval_bar = tqdm(total=len(frag_list), desc=f"Eval {ep}/{EPOCHS}", leave=False, ncols=0)

    @torch.no_grad()
    def evaluate_with_bar(path_local):
        P, Y = [], []
        dset = ds.dataset(path_local, format="parquet")
        for frag in frag_list:
            scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=SCAN_BATCH, use_threads=False)
            for rec in scanner.to_reader():
                X = np.asarray(rec["features"].to_pylist(), dtype=np.float32)
                y = np.asarray(rec["label"].to_pylist(),    dtype=np.float32)
                for i in range(0, X.shape[0], BATCH_SIZE):
                    Xt = torch.from_numpy(X[i:i+BATCH_SIZE]).to(DEVICE, non_blocking=True)
                    P.append(torch.sigmoid(model(Xt)).cpu().numpy().astype(np.float32))
                    Y.append(y[i:i+BATCH_SIZE])
            eval_bar.update(1)
        eval_bar.close()
        if not P: 
            return dict(auc=float("nan"), ap=float("nan"), acc=float("nan"), f1=float("nan"))
        p = np.concatenate(P); y = np.concatenate(Y)
        yhat = (p >= 0.5).astype(int)
        def safe(fn, *a, **k):
            try: return fn(*a, **k)
            except: return float("nan")
        return dict(
            auc=safe(roc_auc_score, y, p),
            ap=safe(average_precision_score, y, p),
            acc=float((yhat==y).mean()),
            f1=safe(f1_score, y, yhat),
        )

    # evaluate and print with flush so output appears even if the UI is busy
    m = evaluate_with_bar(VAL_PATH_LOCAL if USE_VAL else TEST_PATH_LOCAL)
    print(f"Epoch {ep:02d} — AUC {m['auc']:.4f} | AP {m['ap']:.4f} | ACC {m['acc']:.4f} | F1 {m['f1']:.4f}", flush=True)

    if not math.isnan(m["auc"]) and m["auc"] > BEST_AUC + 1e-4:
        BEST_AUC, BAD = m["auc"], 0
        torch.save(model.state_dict(), "/dbfs/models_lstm_indexers_v1/lstm_best.pt")
        print("✔ saved new best", flush=True)
    else:
        BAD += 1
        print(f"No improvement ({BAD}/{PATIENCE})", flush=True)
        if BAD >= PATIENCE:
            print(f"Early stop. Best AUC={BEST_AUC:.4f}", flush=True)
            break

print("Training finished. Best checkpoint: /dbfs/models_lstm_indexers_v1/lstm_best.pt", flush=True)

In [0]:
import numpy as np
import pyarrow.dataset as ds
import torch

def _normalize_dbfs(path: str) -> str:
    # PyArrow needs /dbfs/ local mount, not dbfs:/
    return path.replace("dbfs:/", "/dbfs/")

@torch.no_grad()
def collect_probs(folder: str, model, batch_size: int = 256, scan_batch: int = 8192, device: str = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    folder = _normalize_dbfs(folder)

    dataset = ds.dataset(folder, format="parquet")
    frags = list(dataset.get_fragments())

    P, Y = [], []
    model.eval()

    for frag in frags:
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label"], batch_size=scan_batch)
        for rec in scanner.to_reader():
            X = np.array(rec["features"].to_pylist(), dtype=np.float32)  # (B,96,INPUT_SIZE)
            y = np.array(rec["label"].to_pylist(),    dtype=np.float32)  # (B,)

            n = int(X.shape[0])
            for i in range(0, n, int(batch_size)):
                end = i + int(batch_size)
                Xt = torch.from_numpy(X[i:end]).to(device, non_blocking=True)
                probs = torch.sigmoid(model(Xt)).detach().cpu().numpy().astype(np.float32)
                P.append(probs)
                Y.append(y[i:end])

    if not P:
        return np.empty((0,), dtype=np.float32), np.empty((0,), dtype=np.float32)

    return np.concatenate(P, axis=0), np.concatenate(Y, axis=0)


@torch.no_grad()
def collect_probs_with_ids(folder: str, model, batch_size: int = 256, scan_batch: int = 8192, device: str = None):
    """
    Like collect_probs, but also returns ID and DATE arrays for analysis.
    Returns (probs, labels, ids, dates) with lengths N.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    folder = _normalize_dbfs(folder)

    dataset = ds.dataset(folder, format="parquet")
    frags = list(dataset.get_fragments())

    P, Y, IDS, DATES = [], [], [], []
    model.eval()

    for frag in frags:
        scanner = ds.Scanner.from_fragment(frag, columns=["features","label","ID","DATE"], batch_size=scan_batch)
        for rec in scanner.to_reader():
            X = np.array(rec["features"].to_pylist(), dtype=np.float32)
            y = np.array(rec["label"].to_pylist(),    dtype=np.float32)
            ids   = np.array(rec["ID"].to_pylist())
            dates = np.array(rec["DATE"].to_pylist())

            n = X.shape[0]
            for i in range(0, n, batch_size):
                j = slice(i, i+batch_size)
                Xt = torch.from_numpy(X[j]).to(device, non_blocking=True)
                probs = torch.sigmoid(model(Xt)).detach().cpu().numpy().astype(np.float32)
                P.append(probs)
                Y.append(y[j])
                IDS.append(ids[j])
                DATES.append(dates[j])

    if not P:
        return (np.empty((0,), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
                np.array([], dtype=object),
                np.array([], dtype=object))

    return (np.concatenate(P, axis=0),
            np.concatenate(Y, axis=0),
            np.concatenate(IDS, axis=0),
            np.concatenate(DATES, axis=0))


In [0]:
# --- threshold sweep on TEST ---
import numpy as np, json, builtins
from sklearn.metrics import precision_recall_curve

TEST_PATH_LOCAL = TEST_PATH.replace("dbfs:/","/dbfs/")
p, y = collect_probs(TEST_PATH_LOCAL, model, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, device=DEVICE)

pr, rc, thr = precision_recall_curve(y, p)   # pr, rc length M; thr length M-1
f1 = 2*pr*rc/(pr+rc+1e-12)

# ---- max-F1 threshold ----
j = int(np.nanargmax(f1))
if len(thr) == 0:
    thr_best = 0.5
else:
    # the threshold array is aligned to pr/rc[:-1]; use j-1 (clamped) to avoid OOB
    thr_best = float(thr[builtins.max(j-1, 0)])

# ---- high-recall threshold (e.g., >= 0.90) ----
target_recall = 0.90
if np.any(rc >= target_recall):
    k = int(np.argmax(rc >= target_recall))
    thr_rc = float(thr[builtins.max(k-1, 0)]) if len(thr) > 0 else 0.5
else:
    # fallback: highest available recall (end of curve)
    thr_rc = float(thr[-1]) if len(thr) > 0 else 0.5

def scores_at(t):
    yhat = (p >= t).astype(int)
    TP = int(((y==1)&(yhat==1)).sum()); FP = int(((y==0)&(yhat==1)).sum())
    FN = int(((y==1)&(yhat==0)).sum()); TN = int(((y==0)&(yhat==0)).sum())
    prec = TP/(TP+FP+1e-9); rec = TP/(TP+FN+1e-9)
    f1v = 2*prec*rec/(prec+rec+1e-9)
    acc = (yhat==y).mean()
    return dict(threshold=float(t), precision=float(prec), recall=float(rec),
                f1=float(f1v), acc=float(acc), TP=TP, FP=FP, FN=FN, TN=TN)

best_row = scores_at(thr_best)
hiR_row  = scores_at(thr_rc)

display(spark.createDataFrame([best_row, hiR_row]))
print("Chosen thresholds → maxF1:", thr_best, " | recall≥90%:", thr_rc)


In [0]:
df_sequences = spark.read.table("hive_metastore.default.df_sequences")

In [0]:
df_sequences.selectExpr("size(features) AS timesteps", "size(features[0]) AS features_per_step").distinct().show()


In [0]:
df_sequences.groupBy("label").count().orderBy("label").show()

In [0]:
cutoff_date = "2023-11-30"

df_train = df_sequences.filter(col("DATE") < cutoff_date)
df_test  = df_sequences.filter(col("DATE") >= cutoff_date)

In [0]:
df_test = df_test.repartition(100)
df_train = df_train.repartition(100)

In [0]:
df_test.write.mode("overwrite").parquet("/lstm_test_sequences")


In [0]:
df_train.write.mode("overwrite").parquet("/lstm_train_sequences")


In [0]:
%fs ls

In [0]:
os.listdir("/dbfs/lstm_test_sequences/")

In [0]:
dbutils.fs.rm("/lstm_test_minimal", recurse=True)

In [0]:
df_test_loaded = spark.read.parquet("/lstm_test_sequences")

In [0]:
dbutils.fs.rm("/lstm_train_minimal", recurse=True)


In [0]:
df_train_loaded = spark.read.parquet("/lstm_train_sequences")

In [0]:
display(df_test_loaded)

Databricks data profile. Run in Databricks to view.

In [0]:
#df_test_small = df_test_loaded.select("features", "label").persist()

In [0]:
#df_train_small = df_train_loaded.select("features", "label").persist()

In [0]:
#dbutils.fs.rm("/df_test_loaded", recurse=True)


In [0]:

def _array_depth(df, colname):
    t = df.schema[colname].dataType
    d = 0
    while isinstance(t, ArrayType):
        d += 1
        t = t.elementType
    return d

def ensure_96x19(df, features_col='features', nsteps=96, nfeat=19):
    depth = _array_depth(df, features_col)

    c = col(features_col)
    if depth == 3:
        # features: array<array<array<...>>> (e.g., 96 × 1 × 19) → drop the singleton axis
        # element_at(step, 1) is safer than step[0] if empty lists ever appear
        c = transform(c, lambda step: element_at(step, 1))
    elif depth == 2:
        # already array<array<...>> (e.g., 96 × 19) → keep as-is
        pass
    else:
        raise ValueError(f"Unsupported nesting depth {depth} for `{features_col}`")

    # Cast to double and fill nulls
    c = transform(c, lambda step: transform(step, lambda v: coalesce(v.cast('double'), lit(0.0))))

    clean = df.withColumn(features_col, c)
    clean = clean.filter((size(col(features_col)) == nsteps) &
                         (size(col(features_col)[0]) == nfeat))
    return clean

# Apply (no windowing — each row is already a 96×19 sequence)
train_seq = ensure_96x19(df_train_loaded).select('features', 'label')
test_seq  = ensure_96x19(df_test_loaded ).select('features', 'label')

# Quick sanity check
display(train_seq.select(size('features').alias('T'),
                         size(col('features')[0]).alias('F')).summary())


In [0]:
spark.conf.set("spark.sql.files.maxRecordsPerFile", 400000)
(train_seq.write.mode("overwrite").option("compression", "snappy").parquet("dbfs:/lstm_train_96x19"))
(test_seq.write .mode("overwrite").option("compression", "snappy").parquet("dbfs:/lstm_test_96x19"))


In [0]:
def minibatches_from_parquet(folder, batch_size=256, scan_batch=8192, shuffle_files=True):
    """
    Streams (X, y) minibatches from a Parquet folder.
    X: (B, 96, 19) float32, y: (B,) float32
    """
    dset = ds.dataset(folder, format="parquet")
    frags = list(dset.get_fragments())
    if shuffle_files:
        random.shuffle(frags)

    for frag in frags:
        # Read columns in medium chunks to control RAM
        scanner = ds.Scanner.from_fragment(
            frag,
            columns=["features", "label"],
            batch_size=scan_batch
        )
        for rb in scanner.to_batches():  # pyarrow.RecordBatch
            feats = rb.column("features").to_pylist()     # list of 96×19 lists
            labels = rb.column("label").to_pylist()       # list of scalars

            X = np.asarray(feats, dtype=np.float32)       # (N, 96, 19)
            y = np.asarray(labels, dtype=np.float32)      # (N,)

            # Yield fixed-size minibatches
            n = X.shape[0]
            for i in range(0, n, batch_size):
                j = i + batch_size
                yield X[i:j], y[i:j]

In [0]:
import os, math, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim

import pyarrow.dataset as ds      # <-- this was missing
import pyarrow.parquet as pq  

import torch, torch.nn as nn, torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_SIZE = 19
HIDDEN = 64
LAYERS = 1
DROPOUT = 0.1
LR = 2e-3
EPOCHS = 8
BATCH_SIZE = 256
SCAN_BATCH = 8192
PATIENCE = 3

TRAIN_PATH = "/dbfs/lstm_train_96x19"
TEST_PATH  = "/dbfs/lstm_test_96x19"
SAVE_DIR = "/dbfs/models_lstm"
os.makedirs(SAVE_DIR, exist_ok=True)

class LSTMClf(nn.Module):
    def __init__(self, input_size=19, hidden=64, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=layers,
                            dropout=(dropout if layers > 1 else 0.0),
                            batch_first=True, bidirectional=False)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):                    # x: (B, 96, 19)
        out, (hn, cn) = self.lstm(x)         # hn: (layers, B, H)
        last = hn[-1]                         # (B, H)
        return self.head(last).squeeze(1)     # logits: (B,)

model = LSTMClf(INPUT_SIZE, HIDDEN, LAYERS, DROPOUT).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

@torch.no_grad()
def evaluate(parquet_folder):
    model.eval()
    all_p, all_y = [], []
    for X, y in minibatches_from_parquet(parquet_folder, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=False):
        Xt = torch.from_numpy(X).to(DEVICE)
        logits = model(Xt)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_p.append(probs)
        all_y.append(y)
    if not all_p:  # empty?
        return dict(auc=float("nan"), ap=float("nan"), acc=float("nan"), f1=float("nan"))
    p = np.concatenate(all_p); y = np.concatenate(all_y)
    # Threshold at 0.5 for acc/f1; tune later if needed
    yhat = (p >= 0.5).astype(np.int32)
    # Guard AUC/AP if dataset is single-class (can happen in a shard)
    try: auc = roc_auc_score(y, p)
    except: auc = float("nan")
    try: ap  = average_precision_score(y, p)
    except: ap  = float("nan")
    acc = accuracy_score(y, yhat)
    f1  = f1_score(y, yhat, zero_division=0)
    return dict(auc=auc, ap=ap, acc=acc, f1=f1)

best_auc, bad = -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running, seen = 0.0, 0
    for X, y in minibatches_from_parquet(TRAIN_PATH, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH):
        Xt = torch.from_numpy(X).to(DEVICE)
        yt = torch.from_numpy(y).to(DEVICE)

        opt.zero_grad(set_to_none=True)
        logits = model(Xt)
        loss = loss_fn(logits, yt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        running += float(loss.item()) * yt.size(0)
        seen += yt.size(0)

        # free ASAP
        del Xt, yt, logits, loss

    metrics = evaluate(TEST_PATH)
    denom = seen if seen else 1
    print(f"Epoch {epoch:02d} | loss {running/denom:.5f} | "
      f"AUC {metrics['auc']:.4f} | AP {metrics['ap']:.4f} | "
      f"ACC {metrics['acc']:.4f} | F1 {metrics['f1']:.4f}")

    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "lstm_last.pt"))
    if not math.isnan(metrics["auc"]) and metrics["auc"] > best_auc + 1e-4:
        best_auc, bad = metrics["auc"], 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "lstm_best.pt"))
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"Early stop. Best AUC={best_auc:.4f}")
            break

print("Best checkpoint:", os.path.join(SAVE_DIR, "lstm_best.pt"))


In [0]:
# === Self-contained LSTM evaluation for best checkpoint (fixed for Spark max() clash) ===
import os, numpy as np, torch, torch.nn as nn, builtins
import pyarrow.dataset as ds
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, average_precision_score,
    precision_recall_curve, accuracy_score, f1_score,
    matthews_corrcoef, brier_score_loss, log_loss
)

# ---- Paths & hyperparams (match training) ----
TEST_PATH   = "/dbfs/lstm_test_96x19"
BEST_CKPT   = "/dbfs/models_lstm/lstm_best.pt"
INPUT_SIZE  = 19
N_STEPS     = 96
HIDDEN      = 64
LAYERS      = 1
DROPOUT     = 0.1

SCAN_BATCH  = 4096   # parquet scan chunk
EVAL_BATCH  = 512    # model eval batch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Model ----
class LSTMClf(nn.Module):
    def __init__(self, input_size=19, hidden=64, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden, num_layers=layers,
            dropout=(dropout if layers > 1 else 0.0),
            batch_first=True, bidirectional=False
        )
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 1))
    def forward(self, x):
        _, (hn, _) = self.lstm(x)   # hn: (layers, B, H)
        last = hn[-1]               # (B, H)
        return self.head(last).squeeze(1)

# ---- Load best checkpoint ----
model = LSTMClf(INPUT_SIZE, HIDDEN, LAYERS, DROPOUT).to(DEVICE)
state = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(state, strict=True)
model.eval()
print(f"Loaded checkpoint: {BEST_CKPT}")

# ---- Data streaming ----
@torch.inference_mode()
def collect_probs_and_labels(parquet_folder):
    dset = ds.dataset(parquet_folder, format="parquet")
    all_probs, all_labels = [], []
    for batch in dset.to_batches(columns=["features", "label"], batch_size=SCAN_BATCH):
        feat_py = batch.column("features").to_pylist()
        mask = [ (isinstance(x, (list,tuple)) and len(x)==N_STEPS
                  and isinstance(x[0], (list,tuple)) and len(x[0])==INPUT_SIZE)
                 for x in feat_py ]
        if not any(mask):
            continue
        X_chunk = np.asarray([feat_py[i] for i,ok in enumerate(mask) if ok], dtype=np.float32)
        y_chunk = np.asarray([batch.column("label")[i].as_py() for i,ok in enumerate(mask) if ok], dtype=np.int32)

        n = len(y_chunk)
        for s in range(0, n, EVAL_BATCH):
            Xt = torch.from_numpy(X_chunk[s:s+EVAL_BATCH]).to(DEVICE)
            probs = torch.sigmoid(model(Xt)).detach().cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y_chunk[s:s+EVAL_BATCH])

    if not all_labels:
        return np.empty((0,), dtype=np.int32), np.empty((0,), dtype=np.float32)
    return np.concatenate(all_labels), np.concatenate(all_probs)

y_true, y_prob = collect_probs_and_labels(TEST_PATH)
if y_true.size == 0:
    raise RuntimeError("No samples found or schema mismatch in TEST_PATH.")

# ---- Metrics (threshold-free) ----
def safe(fn, *args, default=np.nan, **kw):
    try: return fn(*args, **kw)
    except Exception: return default

auc   = safe(roc_auc_score, y_true, y_prob)
ap    = safe(average_precision_score, y_true, y_prob)
brier = safe(brier_score_loss, y_true, y_prob)
ll    = safe(log_loss, y_true, np.clip(y_prob, 1e-7, 1-1e-7))

# ---- Thresholded metrics (avoid bare max) ----
def safe_div(n, d):  # avoids Spark max() shadowing
    return n / (d if d > 0 else 1)

def metrics_at_threshold(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    total = tp + tn + fp + fn
    acc  = safe_div(tp + tn, total)
    prec = safe_div(tp, tp + fp)
    rec  = safe_div(tp, tp + fn)
    spec = safe_div(tn, tn + fp)
    f1   = safe_div(2*prec*rec, (prec + rec)) if (prec + rec) > 0 else 0.0
    bal_acc = 0.5 * (rec + spec)
    mcc = safe(matthews_corrcoef, y_true, y_pred, default=0.0)
    return dict(thr=float(thr), tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
                acc=acc, prec=prec, rec=rec, f1=f1, spec=spec, bal_acc=bal_acc, mcc=mcc)

# Fixed 0.5
m05 = metrics_at_threshold(y_true, y_prob, 0.5)

# Best-F1 threshold on THIS set (use a val set in production)
prec, rec, thr = precision_recall_curve(y_true, y_prob)
if thr.size:
    f1s = 2*prec[:-1]*rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thr[best_idx])
else:
    best_thr = 0.5
mbest = metrics_at_threshold(y_true, y_prob, best_thr)

# ---- Print summary ----
print(f"\nSamples: {len(y_true)} | Pos rate: {y_true.mean():.4f}")
print(f"ROC-AUC: {auc:.4f} | PR-AUC(AP): {ap:.4f} | Brier: {brier:.4f} | LogLoss: {ll:.4f}")

print("\nConfusion @ thr=0.50")
print(f"TP={m05['tp']}  FP={m05['fp']}  TN={m05['tn']}  FN={m05['fn']}")
print(f"ACC={m05['acc']:.4f}  PREC={m05['prec']:.4f}  RECALL={m05['rec']:.4f}  "
      f"F1={m05['f1']:.4f}  SPEC={m05['spec']:.4f}  BAL_ACC={m05['bal_acc']:.4f}  MCC={m05['mcc']:.4f}")

print(f"\nBest-F1 threshold on THIS set: {best_thr:.4f}")
print(f"TP={mbest['tp']}  FP={mbest['fp']}  TN={mbest['tn']}  FN={mbest['fn']}")
print(f"ACC={mbest['acc']:.4f}  PREC={mbest['prec']:.4f}  RECALL={mbest['rec']:.4f}  "
      f"F1={mbest['f1']:.4f}  SPEC={mbest['spec']:.4f}  BAL_ACC={mbest['bal_acc']:.4f}  MCC={mbest['mcc']:.4f}")

# (Optional) save confusion matrices to DBFS
try:
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    os.makedirs("/dbfs/tmp", exist_ok=True)

    def save_cm(path, thr):
        y_pred = (y_prob >= thr).astype(np.int32)
        fig, ax = plt.subplots(figsize=(4,4))
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=[0,1], ax=ax)
        ax.set_title(f"Confusion Matrix (thr={thr:.2f})")
        fig.tight_layout(); plt.savefig(path, dpi=200); plt.close(fig)

    save_cm("/dbfs/tmp/cm_thr_0.50.png", 0.5)
    save_cm("/dbfs/tmp/cm_thr_bestF1.png", best_thr)
    print("\nSaved PNGs:")
    print(" - dbfs:/tmp/cm_thr_0.50.png")
    print(" - dbfs:/tmp/cm_thr_bestF1.png")
except Exception as e:
    print(f"(Skipping PNG export) {e}")


In [0]:
import os, matplotlib.pyplot as plt
import matplotlib.image as mpimg

paths = ["/dbfs/tmp/cm_thr_0.50.png", "/dbfs/tmp/cm_thr_bestF1.png"]
for p in paths:
    img = mpimg.imread(p)
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(p))
    plt.show()


In [0]:
from PIL import Image
from IPython.display import display

for p in ["/dbfs/tmp/cm_thr_0.50.png", "/dbfs/tmp/cm_thr_bestF1.png"]:
    display(Image.open(p))

In [0]:
# copy to FileStore (served at /files/…)
dbutils.fs.cp("dbfs:/tmp/cm_thr_0.50.png",    "dbfs:/FileStore/cm_thr_0.50.png",    True)
dbutils.fs.cp("dbfs:/tmp/cm_thr_bestF1.png",  "dbfs:/FileStore/cm_thr_bestF1.png",  True)

displayHTML("""
<div>
  <p><a href="/files/cm_thr_0.50.png" target="_blank">Open cm_thr_0.50.png</a></p>
  <p><a href="/files/cm_thr_bestF1.png" target="_blank">Open cm_thr_bestF1.png</a></p>
  <img src="/files/cm_thr_0.50.png" style="max-width:45%;margin-right:2%"/>
  <img src="/files/cm_thr_bestF1.png" style="max-width:45%"/>
</div>
""")


In [0]:
#df_test_small.repartition(100).write.mode("overwrite").parquet("/lstm_test_minimal")


In [0]:
# df_train_small = df_train_small.persist(StorageLevel.DISK_ONLY)
# _ = df_train_small.count()

In [0]:
# df_train_small.repartition(400).write.mode("overwrite").parquet("/lstm_train_minimal")


In [0]:
# ===== Ultra-lean LSTM eval (no retrain, low RAM) =====
import os, json, time, math, builtins
import numpy as np

# (1) Disable GPU to avoid heavy CUDA memory footprint
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

import torch, torch.nn as nn
from math import sqrt

# ---- paths / small knobs (edit as needed)
TEST_PATH  = "/dbfs/lstm_test_96x19"
BEST_CKPT  = "/dbfs/models_lstm/lstm_best.pt"
OUT_DIR    = "/dbfs/tmp/lstm_best_eval_light"
os.makedirs(OUT_DIR, exist_ok=True)

# model architecture used when you trained
INPUT_SIZE = 19
HIDDEN     = 64
LAYERS     = 1
DROPOUT    = 0.1

# streaming knobs: keep them small to be gentle on memory
BATCH_SIZE = 64
SCAN_BATCH = 2048

# histogram bins for streaming metrics (1000 = good tradeoff)
N_BINS = 1000

# (2) Model definition & load
class LSTMClf(nn.Module):
    def __init__(self, input_size=19, hidden=64, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=layers,
                            dropout=(dropout if layers > 1 else 0.0),
                            batch_first=True, bidirectional=False)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 1))
    def forward(self, x):               # x: (B, 96, 19)
        _, (hn, _) = self.lstm(x)
        last = hn[-1]                   # (B, H)
        return self.head(last).squeeze(1)  # logits (B,)

DEVICE = "cpu"  # force CPU for stability
torch.set_grad_enabled(False)

model = LSTMClf(INPUT_SIZE, HIDDEN, LAYERS, DROPOUT).to(DEVICE)
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.eval()



In [0]:
# (3) Parquet streaming (no Spark collect; pure Arrow)
import pyarrow.dataset as ds
def minibatches_from_parquet(folder, batch_size=64, scan_batch=2048):
    dset = ds.dataset(folder, format="parquet")
    # prefer scanner with small batch_size; fall back if needed
    try:
        scanner = ds.Scanner.from_dataset(dset, batch_size=scan_batch)
        batches_iter = scanner.to_batches()
    except Exception:
        batches_iter = dset.to_batches(batch_size=scan_batch)

    bufX, bufy = [], []
    for rb in batches_iter:
        cols = rb.schema.names
        X = rb.column(cols.index("features")).to_pylist()
        y = rb.column(cols.index("label")).to_pylist()
        for xi, yi in zip(X, y):
            bufX.append(xi)
            bufy.append(float(yi))
            if len(bufX) == batch_size:
                yield np.array(bufX, np.float32), np.array(bufy, np.float32)
                bufX, bufy = [], []
    if bufX:
        yield np.array(bufX, np.float32), np.array(bufy, np.float32)

# (4) Streaming evaluation with histograms (constant RAM)
def stream_eval_hist():
    # histograms of probabilities for pos/neg
    pos_hist = np.zeros(N_BINS, dtype=np.int64)
    neg_hist = np.zeros(N_BINS, dtype=np.int64)

    # also keep running confusion at fixed thr=0.5
    tp = fp = tn = fn = 0

    # bin helper
    def bin_of(p):
        # clamp to [0, 1-eps] so index is in [0, N_BINS-1]
        p = 0.0 if p < 0.0 else (1.0 - 1e-12 if p >= 1.0 else p)
        return int(p * N_BINS)

    # main streaming loop
    total = 0
    with torch.inference_mode():
        for X, y in minibatches_from_parquet(TEST_PATH, BATCH_SIZE, SCAN_BATCH):
            Xt = torch.from_numpy(X).to(DEVICE)
            logits = model(Xt)
            p = torch.sigmoid(logits).cpu().numpy()
            yb = y.astype(np.int32)

            # update histograms & fixed-0.5 confusion
            for yi, pi in zip(yb, p):
                b = bin_of(float(pi))
                if yi == 1: pos_hist[b] += 1
                else:        neg_hist[b] += 1

                pred = 1 if pi >= 0.5 else 0
                if pred == 1 and yi == 1: tp += 1
                elif pred == 1 and yi == 0: fp += 1
                elif pred == 0 and yi == 1: fn += 1
                else: tn += 1

            total += len(yb)

    P = pos_hist.sum()
    N = neg_hist.sum()

    # cumulative from high->low threshold to build PR/ROC & F1
    cum_pos = np.cumsum(pos_hist[::-1])[::-1]
    cum_neg = np.cumsum(neg_hist[::-1])[::-1]

    # arrays per threshold (bin edge)
    tp_arr = cum_pos
    fp_arr = cum_neg
    fn_arr = P - tp_arr
    tn_arr = N - fp_arr

    # precision/recall/f1 across bins
    prec = np.divide(tp_arr, (tp_arr + fp_arr), out=np.zeros_like(tp_arr, dtype=float), where=(tp_arr+fp_arr)!=0)
    rec  = np.divide(tp_arr, (tp_arr + fn_arr), out=np.zeros_like(tp_arr, dtype=float), where=(tp_arr+fn_arr)!=0)
    f1   = np.divide(2*prec*rec, (prec+rec), out=np.zeros_like(prec, dtype=float), where=(prec+rec)!=0)

    # best-F1 bin (if multiple ties, pick highest threshold)
    if f1.size > 0:
        best_idx = int(np.argmax(f1))
    else:
        best_idx = 0
    best_thr = (best_idx) / N_BINS  # left edge of the bin
    # derive best-thr confusion
    tpB = int(tp_arr[best_idx]); fpB = int(fp_arr[best_idx])
    fnB = int(fn_arr[best_idx]); tnB = int(tn_arr[best_idx])

    def _metrics(tp, fp, tn, fn):
        tot = tp+fp+tn+fn
        acc  = (tp+tn)/tot if tot else 0.0
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec  = tp/(tp+fn) if (tp+fn) else 0.0
        spec = tn/(tn+fp) if (tn+fp) else 0.0
        bal  = 0.5*(rec+spec)
        f1   = (2*prec*rec)/(prec+rec) if (prec+rec) else 0.0
        denom = sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)) if (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn) else 0.0
        mcc = ((tp*tn - fp*fn)/denom) if denom else 0.0
        return acc, prec, rec, spec, bal, f1, mcc

    # fixed 0.5 metrics
    acc05, prec05, rec05, spec05, bal05, f105, mcc05 = _metrics(tp, fp, tn, fn)
    # best-F1 metrics
    accB,  precB,  recB,  specB,  balB,  f1B,  mccB  = _metrics(tpB, fpB, tnB, fnB)

    # --- AUC & AP (histogram approximation) ---
    # ROC curve points (FPR, TPR) per bin
    TPR = np.divide(tp_arr, P, out=np.zeros_like(tp_arr, dtype=float), where=P!=0)
    FPR = np.divide(fp_arr, N, out=np.zeros_like(fp_arr, dtype=float), where=N!=0)

    # ensure monotonic by sorting FPR ascending
    order = np.argsort(FPR)
    FPRs  = FPR[order]
    TPRs  = TPR[order]
    auc = float(np.trapz(TPRs, FPRs))  # approx AUC

    # PR curve (rec vs prec) from arrays above
    # sort by recall ascending to integrate
    order_pr = np.argsort(rec)
    rec_s  = rec[order_pr]
    prec_s = prec[order_pr]
    ap = float(np.trapz(prec_s, rec_s))  # approx AP

    # Save confusion matrices
    import matplotlib.pyplot as plt
    def save_cm(tp, fp, tn, fn, title, path):
        import numpy as np
        cm = np.array([[tn, fp],[fn, tp]])
        plt.figure(figsize=(4,4))
        plt.imshow(cm, interpolation='nearest')
        plt.title(title); plt.colorbar()
        ticks = np.arange(2)
        plt.xticks(ticks, ['Pred 0','Pred 1'])
        plt.yticks(ticks, ['True 0','True 1'])
        t = cm.max()/2 if cm.size else 0
        for i in range(2):
            for j in range(2):
                v = int(cm[i,j])
                plt.text(j, i, str(v), ha='center', va='center', color=('white' if v>t else 'black'))
        plt.tight_layout()
        plt.savefig(path, dpi=160, bbox_inches='tight'); plt.close()

    p05   = os.path.join(OUT_DIR, "cm_thr_0.50.png")
    pbest = os.path.join(OUT_DIR, f"cm_thr_{best_thr:.3f}.png")
    save_cm(tp=tp,  fp=fp,  tn=tn,  fn=fn,
            title=f"LSTM (thr=0.50) AUC≈{auc:.3f} F1={f105:.3f}", path=p05)
    save_cm(tp=tpB, fp=fpB, tn=tnB, fn=fnB,
            title=f"LSTM (thr≈{best_thr:.3f}) AUC≈{auc:.3f} F1={f1B:.3f}", path=pbest)

    report = {
        "n_samples": int(P+N),
        "pos_rate": float(P/(P+N)) if (P+N) else 0.0,
        "auc_hist": auc, "ap_hist": ap,  # histogram approximations
        "fixed_0.5": {"tp":int(tp),"fp":int(fp),"tn":int(tn),"fn":int(fn),
                      "acc":acc05,"prec":prec05,"rec":rec05,"spec":spec05,"bal_acc":bal05,"f1":f105,"mcc":mcc05},
        "bestF1":    {"thr": float(best_thr), "tp":int(tpB),"fp":int(fpB),"tn":int(tnB),"fn":int(fnB),
                      "acc":accB,"prec":precB,"rec":recB,"spec":specB,"bal_acc":balB,"f1":f1B,"mcc":mccB},
        "paths": {"cm_fixed": p05, "cm_bestF1": pbest},
        "out_dir": OUT_DIR
    }
    with open(os.path.join(OUT_DIR, "metrics_hist.json"), "w") as f:
        json.dump(report, f, indent=2)
    return report



In [0]:
rep = stream_eval_hist()
print(json.dumps(rep, indent=2))


In [0]:
from PIL import Image
from IPython.display import display
display(Image.open(os.path.join(rep["out_dir"], "cm_thr_0.50.png")))
display(Image.open(os.path.join(rep["out_dir"], "cm_thr_{:.3f}.png".format(rep["bestF1"]["thr"]))))


In [0]:
# ===== LSTM feature importance (streaming, CPU, no retrain) =====
import os, json
import numpy as np

# stay CPU to avoid CUDA memory footprint
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

import torch, torch.nn as nn
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

# --- Paths / knobs ---
TEST_PATH  = "/dbfs/lstm_test_96x19"
BEST_CKPT  = "/dbfs/models_lstm/lstm_best.pt"
OUT_DIR    = "/dbfs/tmp/lstm_best_importance"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_SIZE = 19
HIDDEN     = 64
LAYERS     = 1
DROPOUT    = 0.1

BATCH_SIZE = 64       # keep small = low RAM
SCAN_BATCH = 2048
N_BINS     = 1000     # histogram bins for streaming AUC
FEATURE_NAMES = [f"f{i}" for i in range(19)]  # rename with your real names if you have them

# --- Model (must match what you trained) ---
class LSTMClf(nn.Module):
    def __init__(self, input_size=19, hidden=64, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers=layers,
                            dropout=(dropout if layers > 1 else 0.0),
                            batch_first=True, bidirectional=False)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 1))
    def forward(self, x):                  # x: (B, 96, 19)
        _, (hn, _) = self.lstm(x)
        last = hn[-1]                      # (B, H)
        return self.head(last).squeeze(1)  # logits: (B,)

DEVICE = "cpu"
torch.set_grad_enabled(False)
model = LSTMClf(INPUT_SIZE, HIDDEN, LAYERS, DROPOUT).to(DEVICE)
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.eval()

# --- Stream Parquet as mini-batches (no Spark collect) ---
def minibatches_from_parquet(folder, batch_size=64, scan_batch=2048, limit_record_batches=None):
    dset = ds.dataset(folder, format="parquet")
    try:
        scanner = ds.Scanner.from_dataset(dset, batch_size=scan_batch)
        batches_iter = scanner.to_batches()
    except Exception:
        batches_iter = dset.to_batches(batch_size=scan_batch)

    bufX, bufy = [], []
    seen_rb = 0
    for rb in batches_iter:
        cols = rb.schema.names
        X_list = rb.column(cols.index("features")).to_pylist()
        y_list = rb.column(cols.index("label")).to_pylist()
        for xi, yi in zip(X_list, y_list):
            bufX.append(xi)
            bufy.append(float(yi))
            if len(bufX) == batch_size:
                yield np.array(bufX, np.float32), np.array(bufy, np.float32)
                bufX, bufy = [], []
        seen_rb += 1
        if (limit_record_batches is not None) and (seen_rb >= limit_record_batches):
            break
    if bufX:
        yield np.array(bufX, np.float32), np.array(bufy, np.float32)

# --- Helpers for streaming AUC via histograms ---
def _bin_index(p, n_bins=N_BINS):
    if p < 0.0: p = 0.0
    if p >= 1.0: p = 1.0 - 1e-12
    return int(p * n_bins)

def auc_from_hist(pos_hist, neg_hist):
    P = pos_hist.sum(); N = neg_hist.sum()
    if P == 0 or N == 0:
        return float("nan")
    cum_pos = np.cumsum(pos_hist[::-1])[::-1]
    cum_neg = np.cumsum(neg_hist[::-1])[::-1]
    TPR = cum_pos / P
    FPR = cum_neg / N
    order = np.argsort(FPR)
    return float(np.trapz(TPR[order], FPR[order]))

def build_histogram(pred_iter):
    pos_hist = np.zeros(N_BINS, dtype=np.int64)
    neg_hist = np.zeros(N_BINS, dtype=np.int64)
    for p, y in pred_iter:
        b = _bin_index(p)
        if y == 1: pos_hist[b] += 1
        else:      neg_hist[b] += 1
    return pos_hist, neg_hist

# --- Baseline predictions generator (streaming) ---
def baseline_predictions():
    with torch.inference_mode():
        for X, y in minibatches_from_parquet(TEST_PATH, BATCH_SIZE, SCAN_BATCH):
            p = torch.sigmoid(model(torch.from_numpy(X).to(DEVICE))).cpu().numpy()
            for pi, yi in zip(p, y.astype(np.int32)):
                yield float(pi), int(yi)

# --- Predictions with a single feature permuted within-batch ---
def permuted_predictions(feature_k, mode="shuffle", rng=None):
    if rng is None:
        rng = np.random.default_rng(123)
    with torch.inference_mode():
        for X, y in minibatches_from_parquet(TEST_PATH, BATCH_SIZE, SCAN_BATCH):
            X2 = X.copy()
            if mode == "shuffle":
                idx = rng.permutation(len(X2))
                # permute the k-th feature across the batch (all 96 timesteps)
                X2[:, :, feature_k] = X2[idx, :, feature_k]
            elif mode == "zero":
                X2[:, :, feature_k] = 0.0
            elif mode == "mean":
                # per-batch mean replacement (keeps scale)
                mu = X2[:, :, feature_k].mean()
                X2[:, :, feature_k] = mu
            else:
                raise ValueError("mode must be one of {'shuffle','zero','mean'}")
            p = torch.sigmoid(model(torch.from_numpy(X2).to(DEVICE))).cpu().numpy()
            for pi, yi in zip(p, y.astype(np.int32)):
                yield float(pi), int(yi)

# --- Compute permutation importance as AUC drop ---
def permutation_importance(feature_names=None, mode="shuffle"):
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(INPUT_SIZE)]

    # baseline
    base_pos, base_neg = build_histogram(baseline_predictions())
    base_auc = auc_from_hist(base_pos, base_neg)

    drops = []
    for k in range(INPUT_SIZE):
        pos_k, neg_k = build_histogram(permuted_predictions(k, mode=mode))
        auc_k = auc_from_hist(pos_k, neg_k)
        drop = base_auc - auc_k
        drops.append((feature_names[k], float(drop), float(auc_k)))

    # sort by drop desc
    drops.sort(key=lambda t: t[1], reverse=True)

    # plot
    names = [t[0] for t in drops]
    vals  = [t[1] for t in drops]
    plt.figure(figsize=(10, max(3, 0.35*len(names))))
    plt.barh(range(len(names)), vals)
    plt.yticks(range(len(names)), names)
    plt.xlabel("AUC drop (permutation)")
    plt.title(f"LSTM permutation importance — baseline AUC≈{base_auc:.4f}")
    plt.tight_layout()
    bar_path = os.path.join(OUT_DIR, f"feature_importance_perm_{mode}.png")
    plt.savefig(bar_path, dpi=160, bbox_inches="tight"); plt.close()

    # save json
    out = {
        "baseline_auc": float(base_auc),
        "mode": mode,
        "ranking": [{"feature": n, "auc_drop": d, "auc_permuted": ak} for n, d, ak in drops],
        "plot_path": bar_path,
        "out_dir": OUT_DIR
    }
    with open(os.path.join(OUT_DIR, f"feature_importance_perm_{mode}.json"), "w") as f:
        json.dump(out, f, indent=2)
    return out

# --- Optional: small-sample saliency (grad × input) ---
def saliency_feature_scores(sample_record_batches=32):
    # NOTE: enables grad, keep sample small to avoid RAM issues
    torch.set_grad_enabled(True)
    model.zero_grad(set_to_none=True)
    model.train(False)

    agg = np.zeros((96, INPUT_SIZE), dtype=np.float64)
    seen = 0

    rb_seen = 0
    for X, y in minibatches_from_parquet(TEST_PATH, BATCH_SIZE, SCAN_BATCH):
        if rb_seen >= sample_record_batches:
            break
        rb_seen += 1

        x = torch.from_numpy(X).to(DEVICE)
        x.requires_grad_(True)
        logits = model(x)
        probs = torch.sigmoid(logits)
        # gradient of sum of probs wrt inputs
        torch.sum(probs).backward()
        g = x.grad.detach().cpu().numpy()
        s = np.abs(g * X)  # grad×input saliency
        agg += s.sum(axis=0)  # sum over batch
        seen += X.shape[0]
        model.zero_grad(set_to_none=True)

    torch.set_grad_enabled(False)

    # per-feature score by summing over time
    feat_scores = agg.sum(axis=0)
    # normalize to sum=1 for readability
    if feat_scores.sum() > 0:
        feat_scores = feat_scores / feat_scores.sum()

    # plot
    order = np.argsort(feat_scores)[::-1]
    names = np.array(FEATURE_NAMES)[order]
    vals  = feat_scores[order]
    plt.figure(figsize=(10, max(3, 0.35*len(names))))
    plt.barh(range(len(names)), vals)
    plt.yticks(range(len(names)), names)
    plt.xlabel("normalized saliency (grad×input)")
    plt.title(f"LSTM saliency (sampled batches={sample_record_batches})")
    plt.tight_layout()
    sal_path = os.path.join(OUT_DIR, "feature_saliency_bar.png")
    plt.savefig(sal_path, dpi=160, bbox_inches="tight"); plt.close()

    # also a coarse heatmap across (time, feature)
    plt.figure(figsize=(10, 6))
    plt.imshow(agg, aspect="auto", origin="lower")
    plt.colorbar(label="sum |grad×input|")
    plt.ylabel("time step (0..95)"); plt.xlabel("feature index")
    plt.title("Saliency heatmap (sum over sampled batches)")
    heat_path = os.path.join(OUT_DIR, "feature_saliency_heatmap.png")
    plt.savefig(heat_path, dpi=160, bbox_inches="tight"); plt.close()

    return {
        "scores": {FEATURE_NAMES[i]: float(feat_scores[i]) for i in range(INPUT_SIZE)},
        "bar_path": sal_path,
        "heatmap_path": heat_path,
        "out_dir": OUT_DIR
    }

# === Run permutation importance (shuffle within-batch) ===
perm = permutation_importance(FEATURE_NAMES, mode="shuffle")
print("Permutation importance (top 5):")
for r in perm["ranking"][:5]:
    print(r)

# === (Optional) run saliency on a small sample ===
# sal = saliency_feature_scores(sample_record_batches=24)  # uncomment to run
# print("Top saliency features:", sorted(sal["scores"].items(), key=lambda x: x[1], reverse=True)[:5])

print("Saved:", perm["plot_path"])
# If you ran saliency:
# print("Saved:", sal["bar_path"], sal["heatmap_path"])


In [0]:
# ================= LSTM feature importance (AUC drop) =================
# - baseline metric: AUC via histogram approximation (memory-light)
# - importance per feature f: AUC_baseline - AUC_with_feature_f_mean_imputed
# - inputs: Parquet test set with columns: features (96x19), label
# - outputs: PNG bar chart + JSON with scores

import os, time, json
import numpy as np
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import torch
import matplotlib.pyplot as plt
from datetime import datetime
import builtins

# ---------- paths & params ----------
TEST_PATH = "/dbfs/lstm_test_96x19"
BEST_CKPT = "/dbfs/models_lstm/lstm_best.pt"
OUT_DIR   = f"/dbfs/tmp/lstm_featimp_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_SIZE  = 19
HIDDEN      = 64
LAYERS      = 1
DROPOUT     = 0.1

BATCH_SIZE  = 128       # lower if memory is tight
SCAN_BATCH  = 4096
N_BINS      = 1000      # histogram bins (bigger → slightly more accurate)
DEVICE      = "cpu"     # use "cuda" if you want GPU; CPU is safer on memory

FEATURE_NAMES = [f"f{i+1}" for i in range(INPUT_SIZE)]  # replace if you have names

torch.set_grad_enabled(False)
model_device = torch.device(DEVICE)

# ---------- model ----------
class LSTMClf(torch.nn.Module):
    def __init__(self, input_size=19, hidden=64, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size, hidden, num_layers=layers,
                                  dropout=(dropout if layers>1 else 0.0),
                                  batch_first=True, bidirectional=False)
        self.head = torch.nn.Sequential(torch.nn.LayerNorm(hidden),
                                        torch.nn.Linear(hidden, 1))
    def forward(self, x):                   # x: (B, 96, 19)
        _, (hn, _) = self.lstm(x)          # hn: (layers, B, H)
        last = hn[-1]                       # (B, H)
        return self.head(last).squeeze(1)   # logits: (B,)

model = LSTMClf(INPUT_SIZE, HIDDEN, LAYERS, DROPOUT).to(model_device)
model.load_state_dict(torch.load(BEST_CKPT, map_location=model_device))
model.eval()

# ---------- data utils ----------
def iter_batches(folder, scan_batch=4096):
    dset = ds.dataset(folder, format="parquet")
    # Scanner may not be available on all builds; fall back if needed
    try:
        scanner = ds.Scanner.from_dataset(dset, batch_size=scan_batch)
        yield from scanner.to_batches()
    except Exception:
        yield from dset.to_batches(batch_size=scan_batch)

def minibatches_np(folder, batch_size=256, scan_batch=4096):
    bufX, bufy = [], []
    for rb in iter_batches(folder, scan_batch):
        names = rb.schema.names
        Xi = rb.column(names.index("features")).to_pylist()
        yi = rb.column(names.index("label")).to_pylist()
        for x,y in zip(Xi, yi):
            bufX.append(x)
            bufy.append(float(y))
            if len(bufX) == batch_size:
                yield np.asarray(bufX, np.float32), np.asarray(bufy, np.float32)
                bufX, bufy = [], []
    if bufX:
        yield np.asarray(bufX, np.float32), np.asarray(bufy, np.float32)

def total_rows_from_metadata(folder) -> int | None:
    try:
        dset = ds.dataset(folder, format="parquet")
        n = 0
        for p in dset.files:
            try:
                n += pq.ParquetFile(p).metadata.num_rows
            except Exception:
                return None
        return n
    except Exception:
        return None

# ---------- metrics via histograms ----------
def auc_from_hists(pos_hist, neg_hist):
    P, N = pos_hist.sum(), neg_hist.sum()
    if P == 0 or N == 0: return float("nan")
    neg_cum = np.cumsum(neg_hist)  # ascending bins
    wins = (pos_hist * np.concatenate(([0], neg_cum[:-1]))).sum()
    ties = (pos_hist * neg_hist).sum() * 0.5
    return float((wins + ties) / (P * N))

def run_pass(folder, model, transform=None, batch_size=256, scan_batch=4096, n_bins=1000, device="cpu"):
    pos_hist = np.zeros(n_bins, dtype=np.int64)
    neg_hist = np.zeros(n_bins, dtype=np.int64)

    with torch.inference_mode():
        for X, y in minibatches_np(folder, batch_size, scan_batch):
            if transform is not None:
                X = transform(X)   # expects/returns np.float32, shape (B,96,19)

            Xt = torch.from_numpy(X).to(device)
            p  = torch.sigmoid(model(Xt)).detach().cpu().numpy()
            idx = np.minimum((p * n_bins).astype(int), n_bins-1)

            yb = y.astype(np.int8)
            # accumulate counts per bin for positives/negatives (vectorized by unique bins)
            ub, cnt = np.unique(idx[yb==1], return_counts=True)
            pos_hist[ub] += cnt
            ub, cnt = np.unique(idx[yb==0], return_counts=True)
            neg_hist[ub] += cnt

    return pos_hist, neg_hist

# ---------- compute per-timestep means (96 x 19) for imputation ----------
def compute_means(folder, batch_size=256, scan_batch=4096):
    sums = np.zeros((96, INPUT_SIZE), dtype=np.float64)
    n    = 0
    for X, _ in minibatches_np(folder, batch_size, scan_batch):
        sums += X.sum(axis=0)              # sum over batch → (96,19)
        n    += X.shape[0]
    means = (sums / builtins.max(n, 1)).astype(np.float32)
    return means  # (96, 19)

# ---------- baseline ----------
print("Computing per-timestep means for imputation …")
means = compute_means(TEST_PATH, BATCH_SIZE, SCAN_BATCH)

print("Baseline pass …")
base_pos, base_neg = run_pass(TEST_PATH, model,
                              transform=None,
                              batch_size=BATCH_SIZE,
                              scan_batch=SCAN_BATCH,
                              n_bins=N_BINS,
                              device=model_device)
auc_base = auc_from_hists(base_pos, base_neg)
print(f"Baseline AUC ≈ {auc_base:.6f}")

# ---------- per-feature occlusion (mean-impute across all 96 steps) ----------
def make_transform_for_feature(fid, means):
    mu = means[:, fid].copy()            # (96,)
    def _tfm(X):
        X2 = X.copy()
        X2[:, :, fid] = mu[None, :]
        return X2
    return _tfm

auc_drop = []
t0 = time.time()
for f in range(INPUT_SIZE):
    t_start = time.time()
    tfm = make_transform_for_feature(f, means)
    pos, neg = run_pass(TEST_PATH, model, transform=tfm,
                        batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH,
                        n_bins=N_BINS, device=model_device)
    auc_f = auc_from_hists(pos, neg)
    drop  = auc_base - auc_f
    auc_drop.append(drop)
    dt = time.time() - t_start
    print(f"Feature {f:02d} ({FEATURE_NAMES[f]}): AUC≈{auc_f:.6f}  ΔAUC={drop:+.6f}  ({dt:.1f}s)")

auc_drop = np.array(auc_drop, dtype=float)

# ---------- save bar plot & JSON ----------
order = np.argsort(-auc_drop)   # descending by importance (bigger ΔAUC = more important)
names_sorted = [FEATURE_NAMES[i] for i in order]
vals_sorted  = auc_drop[order]

plt.figure(figsize=(8, builtins.max(4, 0.35*len(names_sorted))))
plt.barh(names_sorted[::-1], vals_sorted[::-1])
plt.title("LSTM Feature Importance — ΔAUC on mean-impute occlusion")
plt.xlabel("AUC drop (baseline − occluded)")
plt.tight_layout()
bar_path = os.path.join(OUT_DIR, "feature_importance_auc_drop.png")
plt.savefig(bar_path, dpi=160, bbox_inches='tight')
plt.close()

report = {
    "baseline_auc": float(auc_base),
    "feature_names": FEATURE_NAMES,
    "auc_drop": auc_drop.tolist(),
    "sorted_indices_desc": order.tolist(),
    "sorted_names_desc": names_sorted,
    "sorted_auc_drop_desc": vals_sorted.tolist(),
    "plot_path": bar_path,
    "out_dir": OUT_DIR,
    "elapsed_sec": float(time.time() - t0),
}
with open(os.path.join(OUT_DIR, "feature_importance.json"), "w") as f:
    json.dump(report, f, indent=2)

print("\n=== Feature importance done ===")
print("Plot:", bar_path)
print("JSON:", os.path.join(OUT_DIR, "feature_importance.json"))

# (Optional) preview right away in the notebook:
# from PIL import Image
# from IPython.display import display
# display(Image.open(bar_path))


In [0]:
from PIL import Image
from IPython.display import display
display(Image.open(bar_path))

f1 → INTENSITY

f2 → TENSION

f3 → H_LIM_I

f4 → H_LIM_T

f5 → MAVERAGE_2H_I

f6 → MAVERAGE_2H_T

f7 → MAVERAGE_1D_I

f8 → MAVERAGE_1D_T

f9 → EVENT_COUNT_I

f10 → EVENT_COUNT_T

f11 → TIME_OVER_LIMIT_I

f12 → TIME_OVER_LIMIT_T

f13 → DAY_OF_WEEK

f14 → DAY_OF_MONTH

f15 → DAY_OF_YEAR

f16 → HOUR_OF_DAY

f17 → ID_INDEX

f18 → AM_PM_INDEX

f19 → CONCELHO_INDEX

In [0]:
features_to_sequence = [
    "INTENSITY", "TENSION", "H_LIM_I", "H_LIM_T",
    "MAVERAGE_2H_I", "MAVERAGE_2H_T", "MAVERAGE_1D_I", "MAVERAGE_1D_T",
    "EVENT_COUNT_I", "EVENT_COUNT_T", "TIME_OVER_LIMIT_I", "TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK", "DAY_OF_MONTH", "DAY_OF_YEAR", "HOUR_OF_DAY",
    "ID_INDEX", "AM_PM_INDEX", "CONCELHO_INDEX"
]